# Compare model sources with dataset-fixer

This notebook accepts standalone `.pt` files, nnU-Net folders, model-bundle ZIPs,
W&B run references/URLs, and a dataset directory or ZIP. Downloads, safe extraction,
metadata reading, caching, and geometry validation are package behavior.

In [ ]:
#!pip uninstall dataset-fixer -y

In [ ]:
import importlib
import importlib.metadata
import importlib.util
import subprocess
import sys

from packaging.specifiers import SpecifierSet

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    # Distribution name -> (import name, required version)
    critical_packages = {
        "numpy": ("numpy", ">=2.5.1,<2.6"),
        "scipy": ("scipy", ">=1.18,<1.19"),
        "scikit-image": ("skimage", ">=0.26,<0.27"),
        "batchgeneratorsv2": ("batchgeneratorsv2", ">=0.3.2,<0.4"),
        "nnunetv2": ("nnunetv2", "==2.8.1"),
        "opencv-python": ("cv2", ">=4.12"),
        "torch": ("torch", ">=2.2"),
        "torchvision": ("torchvision", ">=0.17"),
    }

    def module_loaded(name):
        return any(
            loaded == name or loaded.startswith(f"{name}.")
            for loaded in sys.modules
        )

    conflicts = []

    if module_loaded("dataset_fixer"):
        conflicts.append(
            "dataset_fixer is already imported and cannot be upgraded safely"
        )

    for distribution, (module, requirement) in critical_packages.items():
        try:
            installed = importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            installed = None

        compatible = (
            installed is not None
            and installed in SpecifierSet(requirement)
        )

        # Installing/upgrading is safe if the module has not yet been loaded.
        if not compatible and module_loaded(module):
            conflicts.append(
                f"{distribution}: loaded version {installed!r}, "
                f"but {requirement} is required"
            )

    if conflicts:
        # Stop before pip changes the active environment.
        print(
            "Cannot safely install into this already-used kernel because binary "
            "dependencies requiring replacement have been imported.\n"
            "Start a fresh Colab runtime and run this cell first—before importing "
            "NumPy, Torch, dataset-fixer, Ultralytics, or nnU-Net.\n\n"
            + "\n".join(conflicts)
        )

    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--progress-bar",
        "on",
        "dataset-fixer @ git+https://github.com/mooch443/dataset-fixer.git",
    ]

    print("Installing dataset-fixer before loading scientific modules:")
    print(" ".join(command), flush=True)

    # stdout/stderr are inherited, so the complete pip output appears in the cell.
    subprocess.run(command, check=True)

    importlib.invalidate_caches()

    print(
        "\nInstallation complete."
    )

In [ ]:
!wandb login

In [ ]:
MODEL_SOURCES = [
    {
        "source": (
            "/content/drive/MyDrive/islands/"
            "islands-128-08.08.2026-merged-1class_yolo-128px-yolox.pt"
        ),
        "native_tile_size": 128,
        "upscale_factor": 2,
    },
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/gnsuhtfc",
    "wandb:max-planck-institute-for-animal-behavior/schools-segmentation/0we3e4aq",

    # Bundle ZIP:
    # "/content/drive/MyDrive/models/downloaded-yolo-bundle.zip",

    # Standalone checkpoint with missing metadata:
    # {
    #     "source": "/content/drive/MyDrive/models/best.pt",
    #     "name": "standalone-yolo",
    #     "native_tile_size": 128,
    #     "upscale_factor": 2,
    # },

    # Select a particular file from a W&B run:
    # {
    #     "source": "wandb:entity/project/run-id",
    #     "name": "short-display-name",
    #     "run_file": "exact-bundle-name.zip",  # or "best.pt"
    # },
]

DATASET_SOURCE = (
    "/content/drive/MyDrive/islands/islands-fair-base-sem.zip"
    if IN_COLAB
    else "/Users/tristan/Downloads/island-dataset/islands-fair-base-sem"
)

SPLIT = "val"
INFERENCE = "sahi"
BATCH_SIZE = -1          # Adaptive, always capped at 128
SAHI_OVERLAP = 0.15
WORKERS = 4
SAVE_PREDICTION_PLOTS = True
COMPARISON_DESTINATION = None

# Only metadata or settings that differ for particular resolved models.
MODEL_CONFIGURATION = {
    # "best": {
    #     "task": "semantic",
    #     "native_tile_size": 128,
    #     "upscale_factor": 2,
    #     "input_size": 256,
    #     "batch_size": 16,
    # },
}

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import torch

from dataset_fixer import Dataset, Model

if not MODEL_SOURCES:
    raise ValueError("Add at least one model source to MODEL_SOURCES")

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if (
        getattr(torch.backends, "mps", None) is not None
        and torch.backends.mps.is_available()
    )
    else "cpu"
)

dataset = Dataset.open(DATASET_SOURCE)
models = Model.load_many(MODEL_SOURCES)

print("Resolved models:", models.names)
print("Selected device:", device)

unknown = sorted(set(MODEL_CONFIGURATION) - set(models.names))
if unknown:
    raise KeyError(
        f"MODEL_CONFIGURATION contains unknown models: {unknown}. "
        f"Resolved names are: {models.names}"
    )

common_configuration = {
    "device": device,
    "workers": WORKERS,
    "inference": INFERENCE,
    "batch_size": BATCH_SIZE,
    "sahi_overlap": SAHI_OVERLAP,
}

models = models.configure({
    name: {
        **common_configuration,
        **MODEL_CONFIGURATION.get(name, {}),
    }
    for name in models.names
})

model_descriptions = [model.describe() for model in models]
model_descriptions

In [ ]:
comparison = models.compare(
    dataset,
    split=SPLIT,
    save_prediction_plots=SAVE_PREDICTION_PLOTS,
    destination=COMPARISON_DESTINATION,
    progress=True,
)

comparison

In [ ]:
!zip -r dataset-fixer-cache.zip ./dataset-fixer-cache/
!rsync --progress dataset-fixer-cache.zip /content/drive/MyDrive/islands/dataset-fixer-cache.zip